# Cuaderno de pandas · alquiler vacacional en Mallorca

Datos: Inside Airbnb, Mallorca (CC BY 4.0). Generado y ejecutado por `cuaderno.py`; las salidas son las de la última ejecución.


In [0]:
from pathlib import Path
DATOS = Path("datos")   # listings.csv.gz, calendar.csv.gz y reviews.csv de Inside Airbnb


## 1. Cargar solo lo necesario y con el tipo correcto

Leer 12 de 90 columnas y pasar los textos repetidos a category reduce la memoria. En tablas grandes es la diferencia entre que el proceso quepa o no.

In [1]:
import numpy as np
import pandas as pd

COLS = ["id", "host_id", "neighbourhood_cleansed", "room_type", "accommodates", "bedrooms",
        "bathrooms", "price", "number_of_reviews_ltm", "estimated_revenue_l365d",
        "review_scores_rating", "calculated_host_listings_count"]

completo = pd.read_csv(DATOS / "listings.csv.gz", low_memory=False)
l = pd.read_csv(DATOS / "listings.csv.gz", usecols=COLS)
l_cat = l.astype({"neighbourhood_cleansed": "category", "room_type": "category"})

mb = lambda d: d.memory_usage(deep=True).sum() / 2**20
pd.DataFrame({"columnas": [completo.shape[1], l.shape[1], l_cat.shape[1]],
              "memoria_MB": [mb(completo), mb(l), mb(l_cat)]},
             index=["fichero completo", "12 columnas", "12 columnas + category"]).round(1)

,columnas,memoria_MB
fichero completo,90,74.7
12 columnas,12,3.6
12 columnas + category,12,1.8


## 2. Perfil de calidad antes de calcular nada

Una función reutilizable que dice, por columna, el tipo, cuántos nulos hay y cuántos valores distintos. Aquí aparece que una cuarta parte de los anuncios no publica precio.

In [2]:
def perfil(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos_pct": df.isna().mean().mul(100).round(1),
        "distintos": df.nunique(),
        "ejemplo": df.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None),
    }).sort_values("nulos_pct", ascending=False)

print(f"{len(l)} filas · {l['id'].duplicated().sum()} ids duplicados · {l.shape[1]} columnas")
perfil(l).head(6)

14817 filas · 0 ids duplicados · 12 columnas


,tipo,nulos_pct,distintos,ejemplo
estimated_revenue_l365d,float64,25.4,6336,19176.0
price,str,25.4,6181,$319.60
review_scores_rating,float64,21.7,147,5.0
bathrooms,float64,11.2,28,1.0
bedrooms,float64,3.4,18,3.0
neighbourhood_cleansed,str,0.0,53,Alcúdia


## 3. Limpieza encadenada y con controles

Method chaining con assign y pipe: cada paso se lee de arriba abajo y no deja variables intermedias. Los assert paran el proceso si algo no tiene sentido.

In [3]:
def recortar(df, col, q=(0.01, 0.99)):
    lo, hi = df[col].quantile(list(q))
    return df.assign(**{f"{col}_atipico": ~df[col].between(lo, hi) & df[col].notna()})

limpio = (l
    .rename(columns={"neighbourhood_cleansed": "municipio"})
    .assign(
        precio=lambda d: pd.to_numeric(d["price"].str.replace(r"[$,]", "", regex=True), errors="coerce"),
        tramo=lambda d: pd.cut(d["calculated_host_listings_count"], [0, 1, 4, 19, np.inf],
                               labels=["1", "2-4", "5-19", "20+"]),
        ingreso=lambda d: d["estimated_revenue_l365d"].where(d["estimated_revenue_l365d"] > 0),
    )
    .pipe(recortar, "precio")
    .drop(columns=["price", "estimated_revenue_l365d"]))

assert limpio["id"].is_unique
assert limpio["precio"].dropna().gt(0).all()
limpio[["municipio", "accommodates", "precio", "precio_atipico", "tramo", "ingreso"]].head(5)

,municipio,accommodates,precio,precio_atipico,tramo,ingreso
0,Alcúdia,5,NaN,False,20+,NaN
1,Búger,6,319.60,False,1,19176.0
2,Pollença,2,201.43,False,20+,NaN
3,Sant Llorenç des Cardassar,8,330.71,False,20+,NaN
4,Pollença,5,554.50,False,20+,11090.0


## 4. Agregar con nombre: un resumen por municipio

groupby con agregaciones con nombre: el resultado sale con columnas legibles y listo para un informe. Las medianas, y no las medias, porque el precio tiene una cola larga.

In [4]:
resumen = (limpio
    .groupby("municipio", observed=True)
    .agg(anuncios=("id", "size"),
         precio_mediano=("precio", "median"),
         ingreso_mediano=("ingreso", "median"),
         valoracion=("review_scores_rating", "mean"),
         pct_profesional=("tramo", lambda s: s.isin(["5-19", "20+"]).mean() * 100))
    .sort_values("anuncios", ascending=False))

resumen.head(8).round({"precio_mediano": 0, "ingreso_mediano": 0, "valoracion": 2, "pct_profesional": 1})

,anuncios,precio_mediano,ingreso_mediano,valoracion,pct_profesional
municipio,,,,,
Pollença,2271,427.0,15156.0,4.63,80.2
Alcúdia,1372,427.0,18412.0,4.64,73.5
Santanyí,1014,416.0,16794.0,4.65,68.7
Palma de Mallorca,949,262.0,24233.0,4.71,51.4
Manacor,800,351.0,14618.0,4.74,67.6
Felanitx,676,472.0,15828.0,4.66,73.5
Santa Margalida,631,329.0,16940.0,4.67,68.1
Capdepera,522,383.0,12816.0,4.66,68.4


## 5. Tabla dinámica con totales

pivot_table cruza dos dimensiones como una tabla dinámica de Excel, con la fila y la columna de total (margins).

In [5]:
capacidad = pd.cut(limpio["accommodates"], [0, 2, 4, 6, np.inf], labels=["1-2", "3-4", "5-6", "7+"])
top5 = resumen.index[:5]

(limpio.assign(capacidad=capacidad)
    .query("municipio in @top5")
    .pivot_table(index="municipio", columns="capacidad", values="precio",
                 aggfunc="median", margins=True, margins_name="Total", observed=True)
    .round(0))

capacidad,1-2,3-4,5-6,7+,Total
municipio,,,,,
Alcúdia,193.0,340.0,424.0,648.0,427.0
Manacor,184.0,257.0,347.0,633.0,351.0
Palma de Mallorca,147.0,201.0,371.0,856.0,262.0
Pollença,229.0,299.0,410.0,739.0,427.0
Santanyí,199.0,258.0,456.0,785.0,416.0
Total,187.0,280.0,411.0,724.0,391.0


## 6. Cruzar tablas sin perder filas por el camino

Un merge con validate (si la relación no es la esperada, falla) e indicator, que permite encontrar los huérfanos: noches del calendario de anuncios que ya no están en el listado.

In [6]:
cal = pd.read_csv(DATOS / "calendar.csv.gz", usecols=["listing_id", "date", "available"],
                  parse_dates=["date"])

cruce = cal.merge(limpio[["id", "municipio"]], left_on="listing_id", right_on="id",
                  how="left", validate="many_to_one", indicator=True)

(cruce.groupby("_merge", observed=True)
    .agg(noches=("listing_id", "size"), anuncios=("listing_id", "nunique")))

,noches,anuncios
_merge,,
left_only,30660,84
both,5408205,14817


## 7. Un promedio de promedios no es el total

El error más habitual en un dashboard: promediar los porcentajes o las medianas de cada municipio. Así un pueblo con 10 anuncios pesa lo mismo que uno con 2.000. El dato de Mallorca se calcula sobre los anuncios, no sobre los municipios.

In [7]:
profesional = limpio["tramo"].isin(["5-19", "20+"])

pd.DataFrame({
    "media_de_municipios": [resumen["pct_profesional"].mean(), resumen["precio_mediano"].mean()],
    "calculo_correcto": [profesional.mean() * 100, limpio["precio"].median()],
}, index=["% anuncios de anfitriones con 5+", "precio mediano por noche (€)"]).assign(
    diferencia=lambda d: d["media_de_municipios"] - d["calculo_correcto"]).round(1)

,media_de_municipios,calculo_correcto,diferencia
% anuncios de anfitriones con 5+,64.9,68.5,-3.7
precio mediano por noche (€),411.7,397.6,14.1


## 8. Series temporales: mes, media móvil e interanual

resample agrupa por mes, rolling suaviza la estacionalidad y pct_change(12) compara cada mes con el mismo mes del año anterior, que es la comparación justa en turismo.

In [8]:
rv = pd.read_csv(DATOS / "reviews.csv", usecols=["date"], parse_dates=["date"])

mensual = (rv.set_index("date").resample("MS").size().rename("reseñas")
    .loc["2023-01":"2025-12"].to_frame()
    .rename(index=lambda d: d.strftime("%Y-%m"))
    .assign(media_movil_12m=lambda d: d["reseñas"].rolling(12).mean(),
            interanual_pct=lambda d: d["reseñas"].pct_change(12) * 100))

mensual.tail(6).round(1)

,reseñas,media_movil_12m,interanual_pct
date,,,
2025-07,11129,6623.2,16.7
2025-08,11255,6762.6,17.5
2025-09,11388,6913.3,18.9
2025-10,9684,7030.7,17.0
2025-11,3145,7061.6,13.4
2025-12,1614,7072.2,8.6


## 9. Concentración: ¿cuánto ingreso hace el 10 % de arriba?

qcut parte en deciles y transform calcula el peso de cada anuncio dentro de su municipio sin perder filas. Es la base de un análisis de Pareto.

In [9]:
con_ingreso = limpio.dropna(subset=["ingreso"]).copy()
con_ingreso["decil"] = pd.qcut(con_ingreso["ingreso"].rank(method="first", ascending=False),
                               10, labels=[f"D{i}" for i in range(1, 11)])
con_ingreso["peso_en_municipio"] = (con_ingreso["ingreso"]
    / con_ingreso.groupby("municipio", observed=True)["ingreso"].transform("sum"))

(con_ingreso.groupby("decil", observed=True)["ingreso"].sum()
    .div(con_ingreso["ingreso"].sum()).mul(100)
    .to_frame("pct_del_ingreso")
    .assign(acumulado=lambda d: d["pct_del_ingreso"].cumsum())
    .head(4).round(1))

,pct_del_ingreso,acumulado
decil,,
D1,34.8,34.8
D2,18.0,52.8
D3,13.0,65.8
D4,9.9,75.8


## 10. Vectorizar donde importa, y medirlo

Clasificar cada anuncio en un segmento recorriendo filas con apply(axis=1) frente a hacerlo por columnas con np.select. Mismo resultado, comprobado con assert. Ojo: con textos, apply puede ser igual de rápido; por eso mido antes de optimizar.

In [10]:
import time

base = limpio.dropna(subset=["precio"])

def fila_a_fila(d):
    def segmento(f):
        if f["precio"] >= 600 and f["accommodates"] >= 7:
            return "lujo grupos"
        if f["precio"] >= 600:
            return "lujo"
        if f["accommodates"] >= 7:
            return "grupos"
        return "estandar"
    return d.apply(segmento, axis=1)

def por_columnas(d):
    caro, grande = d["precio"].ge(600), d["accommodates"].ge(7)
    return pd.Series(np.select([caro & grande, caro, grande], ["lujo grupos", "lujo", "grupos"],
                               default="estandar"), index=d.index)

tiempos = {}
for nombre, f in [("apply(axis=1)", fila_a_fila), ("np.select", por_columnas)]:
    t0 = time.perf_counter()
    resultado = f(base)
    tiempos[nombre] = (time.perf_counter() - t0) * 1000

assert fila_a_fila(base).equals(por_columnas(base))
pd.Series(tiempos, name="milisegundos").to_frame().assign(
    veces_mas_lento=lambda d: d["milisegundos"] / d["milisegundos"].min()).round(1)

,milisegundos,veces_mas_lento
apply(axis=1),40.0,34.6
np.select,1.2,1.0


## 11. Cuadrar pandas contra SQL

La misma métrica calculada por dos caminos debe dar lo mismo. sqlite3 viene con Python: cargo la tabla, repito el cálculo en SQL y comparo con assert_frame_equal.

In [11]:
import sqlite3

con = sqlite3.connect(":memory:")
limpio[["municipio", "precio", "ingreso"]].astype({"municipio": str}).to_sql("anuncios", con, index=False)

sql = pd.read_sql("""
    SELECT municipio, COUNT(*) AS anuncios, AVG(precio) AS precio_medio, SUM(ingreso) AS ingreso_total
    FROM anuncios GROUP BY municipio ORDER BY municipio
""", con).set_index("municipio")

pandas_ = (limpio.astype({"municipio": str}).groupby("municipio")
    .agg(anuncios=("id", "size"), precio_medio=("precio", "mean"), ingreso_total=("ingreso", "sum")))

pd.testing.assert_frame_equal(sql, pandas_, check_dtype=False)
print(f"Cuadre correcto en {len(sql)} municipios y 3 métricas")

Cuadre correcto en 53 municipios y 3 métricas
